In [ ]:
import sys
sys.path.append("..")   # add main_folder to path


import geopandas as gpd
import itertools
import pyarrow.dataset as pds
from tqdm import tqdm
import xarray as xr
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib.pyplot as plt
from typing import List, Optional
import pyarrow.compute as pc
import tomllib



from src.bias_correction.bias_correction import correct_bias_with_era5_and_save
from src.bias_correction.bias_correction_utils import (get_era5_benchmark, 
                                                       get_histo_sim_from_benchmark)



In [ ]:
def plot_variable_distributions_by_period(
    df: pd.DataFrame,
    date_col: str = "date",
    variable_cols: Optional[List[str]] = None,  # only for wide format
    long_format: bool = False,                  # set True if df already has 'variable' and 'value'
    value_col: str = "value",
    variable_name_col: str = "variable",
    dropna: bool = True,
    nbins: int = 40,
    alpha: float = 0.5,
    figsize=(6, 4),
    save_dir: Optional[str] = None,             # e.g. "figs"; if None, don't save
):
    """
    Compare distributions of 4 climate variables across three time periods:
      - 2025–2049 inclusive
      - 2050–2074 inclusive
      - >= 2075
    Assumes multiple locations; all locations are pooled within each period.
    Creates one figure per variable with overlaid histograms for the 3 periods.
    """

    # 1) Parse dates and build period labels
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # Define period edges
    p1_start = pd.Timestamp("1980-01-01")
    p1_end   = pd.Timestamp("2049-12-31")
    p2_start = pd.Timestamp("2050-01-01")
    p2_end   = pd.Timestamp("2074-12-31")
    p3_start = pd.Timestamp("2075-01-01")

    def _label_period(ts: pd.Timestamp) -> str:
        if pd.isna(ts):
            return "Unknown"
        if p1_start <= ts <= p1_end:
            return "2025–2049"
        if p2_start <= ts <= p2_end:
            return "2050–2074"
        if ts >= p3_start:
            return "≥ 2075"
        return "< 2025"  # falls before analysis window; we’ll drop later

    out["period"] = out[date_col].map(_label_period)

    # Keep only the three requested periods
    out = out[out["period"].isin(["2025–2049", "2050–2074", "≥ 2075"])]

    # 2) Reshape to long if needed
    if not long_format:
        if variable_cols is None:
            # Try to infer: pick non-date numeric columns (you can pass explicit list instead)
            variable_cols = [c for c in out.columns
                             if c not in {date_col, "period"} and np.issubdtype(out[c].dtype, np.number)]
        long = out.melt(
            id_vars=[date_col, "period"],
            value_vars=variable_cols,
            var_name=variable_name_col,
            value_name=value_col
        )
    else:
        # Expect columns: date_col, variable_name_col, value_col (+ others allowed)
        long = out[[date_col, "period", variable_name_col, value_col]].copy()

    # 3) Clean values
    if dropna:
        long = long.dropna(subset=[value_col])

    # 4) Plot – one figure per variable, overlay the three periods
    variables = long[variable_name_col].unique()

    for var in variables:
        sub = long[long[variable_name_col] == var]

        # Prepare data arrays per period
        data_p1 = sub.loc[sub["period"] == "2025–2049", value_col].to_numpy()
        data_p2 = sub.loc[sub["period"] == "2050–2074", value_col].to_numpy()
        data_p3 = sub.loc[sub["period"] == "≥ 2075",     value_col].to_numpy()

        # Common bins across periods for fair comparison
        # Use combined min/max with small padding
        all_vals = np.concatenate([x for x in [data_p1, data_p2, data_p3] if x.size > 0]) \
                   if (data_p1.size + data_p2.size + data_p3.size) > 0 else np.array([])
        if all_vals.size == 0:
            print(f"[warn] No data to plot for variable '{var}'. Skipping.")
            continue
        vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
        if vmin == vmax:
            # Avoid degenerate bins
            vmin, vmax = vmin - 0.5, vmax + 0.5
        bins = np.linspace(vmin, vmax, nbins)

        # Create figure
        plt.figure(figsize=figsize)
        # Overlaid histograms (density normalized)
        if data_p1.size:
            plt.hist(data_p1, bins=bins, density=True, alpha=alpha, label="2025–2049")
        if data_p2.size:
            plt.hist(data_p2, bins=bins, density=True, alpha=alpha, label="2050–2074")
        if data_p3.size:
            plt.hist(data_p3, bins=bins, density=True, alpha=alpha, label="≥ 2075")

        plt.title(f"Distribution of {var} by period")
        plt.xlabel(var)
        plt.ylabel("Density")
        plt.legend()
        plt.tight_layout()

        if save_dir is not None:
            import os
            os.makedirs(save_dir, exist_ok=True)
            fname = os.path.join(save_dir, f"{var}_distribution_by_period.png")
            plt.savefig(fname, dpi=150)
        # Show each figure separately if running interactively
        # plt.show()

    print("Done. Created one histogram figure per variable.")

In [ ]:
config_path = "../config.toml"
with open(
    config_path,
    "rb",
) as f:  # Open the file in binary mode
    config_files = tomllib.load(f)

In [ ]:
main_config = config_files["main_params"]
gen_config = config_files["generation"]
intens_config = config_files["intensification"]
seeds = list(range(gen_config["n_seeds"]))

catherina_fit_path = ".." / Path(gen_config["fit_dir"]) / "Catherina_fit.db"
cyclones_dir = Path(gen_config["synthetic_tracks_dir"])

data_dir = ".." / Path(main_config["data_dir"])


# Climate variables
cmip_dir = ".." / Path(main_config["data_dir"]) / "cmip6_data_with_hurs/"

ne_10m_coastline_zip = ".." / Path(main_config["data_dir"]) / "ne_10m_coastline.zip"
ne_10m_land_zip = ".." /Path(main_config["data_dir"]) / "ne_10m_land.zip"
model_exp_pbar = tqdm(
    list(itertools.product(main_config["models"], main_config["experiments"]))
)
land = gpd.read_file(ne_10m_land_zip).union_all()

In [ ]:
model="ACCESS-CM2"
experiment="ssp585"

clim_ds = xr.open_zarr(cmip_dir / model / experiment)
model_exp_pbar.set_postfix(model=model, experiment=experiment)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def _wrap_to_180(a):
    # Map longitudes to [-180, 180)
    return ((np.asarray(a, dtype=float) + 180.0) % 360.0) - 180.0

def _split_dateline_segments(lons, lats):
    """Return list of (lon_seg, lat_seg) arrays, split where |Δlon| > 180°."""
    lons = np.asarray(lons, dtype=float)
    lats = np.asarray(lats, dtype=float)

    # Ensure plotting-friendly longitudes
    lons = _wrap_to_180(lons)

    # Find big jumps in longitude between consecutive points
    jumps = np.abs(np.diff(lons))
    breaks = np.where(jumps > 180.0)[0] + 1

    lon_segs = np.split(lons, breaks)
    lat_segs = np.split(lats, breaks)
    return list(zip(lon_segs, lat_segs))

def plot_cyclone_tracks_map(df, sid_col="SID", lat_col="lat", lon_col="lon", time_col=None,
                            show_legend=False, with_markers=False):
    fig, ax = plt.subplots(figsize=(12, 6),
                           subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_global()
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.gridlines(draw_labels=True)

    # One color per SID is optional; avoid clutter if many storms
    for sid, group in df.groupby(sid_col, sort=False):
        g = group
        if time_col is not None and time_col in g.columns:
            g = g.sort_values(time_col)

        lon_vals = g[lon_col].to_numpy()
        lat_vals = g[lat_col].to_numpy()

        for x, y in _split_dateline_segments(lon_vals, lat_vals):
            if len(x) < 2:
                continue
            ax.plot(x, y,
                    transform=ccrs.PlateCarree(),
                    linewidth=1.4, alpha=0.9)
            if with_markers:
                ax.plot(x, y,
                        transform=ccrs.PlateCarree(),
                        marker="o", markersize=2, linestyle="None", alpha=0.9)

        if show_legend:
            ax.plot([], [], label=str(sid))  # dummy handle

    ax.set_title("Cyclone Tracks (dateline-safe)")
    if show_legend:
        ax.legend(title=sid_col, fontsize=8, loc="upper right", bbox_to_anchor=(1.2, 1))
    plt.tight_layout()
    plt.show()



In [ ]:
data_dir = Path(main_config["data_dir"])

tracks_with_env_path = (
        '..' / data_dir/ f"catherina_ssp585/tracks_with_env4/{model}/{experiment}/"
    )

benchmark_path = '..' / data_dir / "bias_correction/ERA5_benchmark_100tracks_bias_corrections.csv"
clim_obs_histo = get_era5_benchmark(benchmark_path=benchmark_path)
clim_sim_histo_ds = xr.open_zarr(
    '..' / data_dir / f"cmip6_data_with_hurs/{model}/historical", chunks="auto"
).drop_vars('hur').rename({"hurs": "hur"}) # TODO: move rename to cmip6_pangeo.py
clim_sim_histo = get_histo_sim_from_benchmark(
    benchmark=clim_obs_histo, histo_clim_ds=clim_sim_histo_ds
)
clim_sim_histo = clim_sim_histo.assign(
    MSLP=clim_sim_histo["MSLP"] / 100.0,
    thermo_eff=((clim_sim_histo["SST"] + 273.15) - clim_sim_histo["T_strat"])
    / (clim_sim_histo["SST"] + 273.15),
)

tracks_with_env_ds = pds.dataset(
    tracks_with_env_path, format="parquet", partitioning="hive"
)

for seed in range(100): #range(gen_config["n_seeds"]):
    correct_bias_with_era5_and_save(
        seeds=[seed],
        tracks_with_env_ds=tracks_with_env_ds,
        clim_obs_histo=clim_obs_histo,
        clim_sim_histo=clim_sim_histo,
        model=model,
        experiment=experiment,
        save_dir='..' / data_dir / "catherina_ssp585",
    )

In [ ]:
data_dir = Path(main_config["data_dir"])

tracks_with_env_path = (
        '..' / data_dir/ f"catherina_test6_to_delete/tracks_with_env/{model}/{experiment}/"
    )

benchmark_path = '..' / data_dir / "bias_correction/ERA5_benchmark_100tracks_bias_corrections.csv"
clim_obs_histo = get_era5_benchmark(benchmark_path=benchmark_path)
clim_sim_histo_ds = xr.open_zarr(
    '..' / data_dir / f"cmip6_data_with_hurs/{model}/historical", chunks="auto"
).drop_vars('hur').rename({"hurs": "hur"}) # TODO: move rename to cmip6_pangeo.py
clim_sim_histo = get_histo_sim_from_benchmark(
    benchmark=clim_obs_histo, histo_clim_ds=clim_sim_histo_ds
)
clim_sim_histo = clim_sim_histo.assign(
    MSLP=clim_sim_histo["MSLP"] / 100.0,
    thermo_eff=((clim_sim_histo["SST"] + 273.15) - clim_sim_histo["T_strat"])
    / (clim_sim_histo["SST"] + 273.15),
)

tracks_with_env_ds = pds.dataset(
    tracks_with_env_path, format="parquet", partitioning="hive"
)


In [ ]:

scanner = pds.Scanner.from_dataset(
        tracks_with_env_ds,
        filter=(
            (pc.field("seed").isin([1]))
        )
    ).to_table()

tracks = pds.dataset(scanner).to_table().to_pandas()

In [ ]:
min_steps = tracks.groupby("SID")["step"].min()

# Find seeds where min step is not 0
bad_seeds = min_steps[min_steps != 0].index.tolist()

print(bad_seeds)

In [ ]:
tracks.loc[lambda row:row['SID']==162209867]

In [ ]:
corrected_tracks= pds.dataset(
            Path(main_config["data_dir"])
                /f"catherina_test4/corrected_tracks/{model}/{experiment}",
            format="parquet",
            partitioning="hive",
        )

In [ ]:
corrected_tracks.to_table().to_pandas().columns

In [ ]:
corrected_tracks.to_table().to_pandas().loc[lambda row:row['SID']==3112942932,["SST","T_strat","MSLP","nshr","distance_track_env",'lat_left','lat_right']]

In [ ]:
corrected_tracks.to_table().to_pandas().columns

In [ ]:
plot_variable_distributions_by_period(
    corrected_tracks.to_table().to_pandas(),
    date_col="datetime",
    variable_cols=["SST","T_strat", "MSLP", "nshr"],  # your 4 variables (wide format)
    long_format=False,          # set True if your df already has 'variable' and 'value'
    nbins=75,
    save_dir=None               # or "figs"
)
